# Homework #5: Model Deployment

## Project Goal
In this assignment, I build two predictive models using the F1 dataset, track both models in MLflow, and store each model's predictions in separate tables in my own database.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [0]:
df_pit = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True, inferSchema=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)

In [0]:
display(df_pit)
display(df_races)
display(df_results)

In [0]:
df_model = (
    df_pit
    .join(
        df_races.select("raceId", "year", "round"),
        on="raceId",
        how="left"
    )
    .join(
        df_results.select("raceId", "driverId", "grid", "positionOrder", "constructorId"),
        on=["raceId", "driverId"],
        how="left"
    )
    .select(
        "milliseconds",
        "stop",
        "lap",
        "driverId",
        "raceId",
        "year",
        "round",
        "grid",
        "positionOrder",
        "constructorId"
    )
    .dropna()
)

In [0]:
display(df_model)

In [0]:
pdf = df_model.toPandas()

X = pdf.drop(columns=["milliseconds"])
y = pdf["milliseconds"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [0]:
spark.sql("SHOW SCHEMAS").show(truncate=False)

In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

In [0]:
spark.sql("SELECT current_catalog()").show(truncate=False)

In [0]:
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
print("Original username:", username)

schema_name = username.split("@")[0].replace(".", "_").replace("-", "_")
print("Schema name:", schema_name)

In [0]:
spark.range(5).write.mode("overwrite").saveAsTable("workspace.default.test_nl2994_table")

In [0]:
spark.sql("SELECT * FROM workspace.default.test_nl2994_table").show()

In [0]:
X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

prediction_base = X_test_reset[["raceId", "driverId"]].copy()
prediction_base["actual_milliseconds"] = y_test_reset

prediction_base.head()

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.rf_predictions_nl2994")
spark.sql("DROP TABLE IF EXISTS workspace.default.gbr_predictions_nl2994")

In [0]:
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
mlflow.set_experiment(f"/Users/{username}/homework5_model_deployment")

In [0]:
with mlflow.start_run(run_name="random_forest_model") as run:
    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42
    )

    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_test)

    rf_mse = mean_squared_error(y_test, rf_preds)
    rf_rmse = np.sqrt(rf_mse)
    rf_mae = mean_absolute_error(y_test, rf_preds)
    rf_r2 = r2_score(y_test, rf_preds)

    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("min_samples_split", 2)
    mlflow.log_param("min_samples_leaf", 1)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("mse", rf_mse)
    mlflow.log_metric("rmse", rf_rmse)
    mlflow.log_metric("mae", rf_mae)
    mlflow.log_metric("r2", rf_r2)

    mlflow.sklearn.log_model(rf, "random_forest_model")

    rf_fi = pd.DataFrame({
        "feature": X_train.columns,
        "importance": rf.feature_importances_
    }).sort_values("importance", ascending=False)

    rf_fi_file = "rf_feature_importance.csv"
    rf_fi.to_csv(rf_fi_file, index=False)
    mlflow.log_artifact(rf_fi_file)

    rf_residuals = y_test - rf_preds
    plt.figure(figsize=(8, 5))
    plt.scatter(rf_preds, rf_residuals, alpha=0.5)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.title("Random Forest Residual Plot")
    plt.tight_layout()
    rf_plot_file = "rf_residual_plot.png"
    plt.savefig(rf_plot_file)
    plt.close()
    mlflow.log_artifact(rf_plot_file)

    print("Random Forest run complete")
    print("RMSE:", rf_rmse)
    print("R2:", rf_r2)

In [0]:
rf_predictions_pd = prediction_base.copy()
rf_predictions_pd["predicted_milliseconds"] = rf_preds
rf_predictions_pd["model_name"] = "RandomForestRegressor"

rf_predictions_spark = spark.createDataFrame(rf_predictions_pd)
rf_predictions_spark.write.mode("overwrite").saveAsTable("workspace.default.rf_predictions_nl2994")

In [0]:
spark.sql("SELECT * FROM workspace.default.rf_predictions_nl2994 LIMIT 10").show()

In [0]:
with mlflow.start_run(run_name="gradient_boosting_model") as run:
    gbr = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    gbr.fit(X_train, y_train)
    gbr_preds = gbr.predict(X_test)

    gbr_mse = mean_squared_error(y_test, gbr_preds)
    gbr_rmse = np.sqrt(gbr_mse)
    gbr_mae = mean_absolute_error(y_test, gbr_preds)
    gbr_r2 = r2_score(y_test, gbr_preds)

    mlflow.log_param("model_type", "GradientBoostingRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 3)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("mse", gbr_mse)
    mlflow.log_metric("rmse", gbr_rmse)
    mlflow.log_metric("mae", gbr_mae)
    mlflow.log_metric("r2", gbr_r2)

    mlflow.sklearn.log_model(gbr, "gradient_boosting_model")

    gbr_fi = pd.DataFrame({
        "feature": X_train.columns,
        "importance": gbr.feature_importances_
    }).sort_values("importance", ascending=False)

    gbr_fi_file = "gbr_feature_importance.csv"
    gbr_fi.to_csv(gbr_fi_file, index=False)
    mlflow.log_artifact(gbr_fi_file)

    gbr_residuals = y_test - gbr_preds
    plt.figure(figsize=(8, 5))
    plt.scatter(gbr_preds, gbr_residuals, alpha=0.5)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.title("Gradient Boosting Residual Plot")
    plt.tight_layout()
    gbr_plot_file = "gbr_residual_plot.png"
    plt.savefig(gbr_plot_file)
    plt.close()
    mlflow.log_artifact(gbr_plot_file)

    print("Gradient Boosting run complete")
    print("RMSE:", gbr_rmse)
    print("R2:", gbr_r2)

In [0]:
gbr_predictions_pd = prediction_base.copy()
gbr_predictions_pd["predicted_milliseconds"] = gbr_preds
gbr_predictions_pd["model_name"] = "GradientBoostingRegressor"

gbr_predictions_spark = spark.createDataFrame(gbr_predictions_pd)
gbr_predictions_spark.write.mode("overwrite").saveAsTable("workspace.default.gbr_predictions_nl2994")

In [0]:
spark.sql("SELECT * FROM workspace.default.gbr_predictions_nl2994 LIMIT 10").show()

## Model Comparison

In this assignment, I built two predictive models using the F1 dataset: a Random Forest Regressor and a Gradient Boosting Regressor. For each model, I logged hyperparameters, the model itself, four evaluation metrics, and two artifacts in MLflow.

I also stored the predictions from each model in separate tables in the database:
- `workspace.default.rf_predictions_nl2994`
- `workspace.default.gbr_predictions_nl2994`

This setup demonstrates both model tracking and model deployment into a database environment.

## Note on Database Storage

I was not able to create a new schema in the `workspace` catalog due to permission restrictions. Therefore, I stored my model prediction tables in `workspace.default` using unique table names with my identifier to keep them separate from other users' work.

## Conclusion

Both models were successfully trained, tracked in MLflow, and deployed by storing their predictions in database tables. Based on the evaluation metrics, the Random Forest model performed better than the Gradient Boosting model, with a lower RMSE and a higher R².